In [1]:
# 05_metamorphic_relations.ipynb
# Data-driven illustration of the four metamorphic relation families. For each family
# we plot directors' Stage-1 source rating (x) against the follow-up rating (y) on the
# simulated data, shade the admissible region implied by the relation, and mark the
# judgments that violate it. This grounds the relation concept in real data and shows
# the observed violation rate per family. Figures greyscale; no paths printed.
import numpy as np, pandas as pd, warnings
from pathlib import Path
warnings.filterwarnings("ignore")
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt, seaborn as sns
sns.set_theme(style="whitegrid", context="paper")
plt.rcParams.update({"savefig.dpi": 600, "font.size": 11,
                     "axes.edgecolor": "black", "axes.linewidth": 0.8})
DATA = Path("..")/"data"; FIG = Path("..")/"results"/"figures"; TAB = Path("..")/"results"/"tables"
FIG.mkdir(parents=True, exist_ok=True); TAB.mkdir(parents=True, exist_ok=True)
def save(fig, name):
    fig.savefig(FIG/f"{name}.png", dpi=600, bbox_inches="tight")
    fig.savefig(FIG/f"{name}.pdf", bbox_inches="tight"); plt.close(fig)

L = pd.read_csv(DATA/"respondent_long.csv")
s1 = L[L.stage == 1].copy()                 # relations are assessed on Stage-1 judgments
rng = np.random.default_rng(7)
s1["xj"] = s1.rating_source + rng.uniform(-0.18, 0.18, len(s1))
s1["yj"] = s1.rating_follow + rng.uniform(-0.18, 0.18, len(s1))

# admissible region per family (as operationalized in the simulation)
#   monotonicity: y <= x   |  equivalence: y == x  |  ordering/composition: y >= x
REGION = {"monotonicity": "lower", "equivalence": "diag",
          "ordering": "upper", "composition": "upper"}
CONSTR = {"monotonicity": r"$y(T(x)) \leq y(x)$", "equivalence": r"$y(T(x)) = y(x)$",
          "ordering": r"$y(T(x)) \geq y(x)$", "composition": r"$y(\mathrm{comp}) \geq \max$ parts"}
order = ["monotonicity", "equivalence", "ordering", "composition"]
lims = (0.4, 7.6)

fig, axes = plt.subplots(2, 2, figsize=(8.6, 8.2))
for ax, rel in zip(axes.ravel(), order):
    g = s1[s1.relation == rel]
    # shaded admissible region
    if REGION[rel] == "lower":
        ax.fill([lims[0], lims[1], lims[1]], [lims[0], lims[0], lims[1]], color="#000000", alpha=0.06, zorder=0)
    elif REGION[rel] == "upper":
        ax.fill([lims[0], lims[0], lims[1]], [lims[0], lims[1], lims[1]], color="#000000", alpha=0.06, zorder=0)
    ax.plot(lims, lims, color="#777777", lw=1.0, ls="--", zorder=1)       # y = x reference
    ok = g[g.violation == 0]; vio = g[g.violation == 1]
    ax.scatter(ok.xj, ok.yj, s=16, facecolor="#9a9a9a", edgecolor="#555555",
               linewidth=0.3, alpha=0.45, zorder=2, label="admissible")
    ax.scatter(vio.xj, vio.yj, s=26, color="#000000", marker="x",
               linewidth=1.1, alpha=0.8, zorder=3, label="violation")
    vr = 100 * g.violation.mean()
    ax.set_title(f"{rel.capitalize()}:  {CONSTR[rel]}", fontsize=11)
    ax.text(0.03, 0.95, f"violations: {vr:.0f}%", transform=ax.transAxes,
            va="top", ha="left", fontsize=10,
            bbox=dict(boxstyle="round,pad=0.25", fc="white", ec="#777777", lw=0.6))
    ax.set_xlim(lims); ax.set_ylim(lims)
    ax.set_xticks(range(1, 8)); ax.set_yticks(range(1, 8))
    ax.set_xlabel("Source rating  y(x)"); ax.set_ylabel("Follow-up rating  y(T(x))")
    ax.set_aspect("equal")
handles, labels = axes[0, 0].get_legend_handles_labels()
fig.legend(handles, labels, loc="lower center", ncol=2, frameon=True,
           edgecolor="black", bbox_to_anchor=(0.5, -0.02))
fig.tight_layout(rect=[0, 0.03, 1, 1])
save(fig, "fig08_metamorphic_relations")

# companion table: violation rate by family and stage
rows = []
for rel in order:
    for st in (1, 2):
        gg = L[(L.relation == rel) & (L.stage == st)]
        rows.append(dict(relation=rel, stage=st, violation_rate=round(gg.violation.mean(), 3), n=len(gg)))
pd.DataFrame(rows).to_csv(TAB/"table07_violation_by_relation.csv", index=False)

print("relations figure built from", len(s1), "Stage-1 judgment pairs")
print(pd.DataFrame(rows).pivot(index="relation", columns="stage", values="violation_rate"))
print("figure: fig08  | table: table07")


relations figure built from 6984 Stage-1 judgment pairs
stage             1      2
relation                  
composition   0.249  0.169
equivalence   0.732  0.664
monotonicity  0.153  0.085
ordering      0.167  0.086
figure: fig08  | table: table07
